# RSNA screening case study: density-stratified AI performance
Inputs (Add Input): the competition **RSNA Screening Mammography Breast Cancer Detection** (for `train.csv`) and the dataset
**theoviel/rsna-breast-cancer-512-pngs**. Accelerator GPU T4 x2, Internet on. Save & Run All.
Plan fixed in advance (docs/rsna_analysis_plan.md): EfficientNet-B0, 512 px, 5-fold patient-level CV, 10 epochs,
out-of-fold breast scores, operating points at 10% and 5% false-positive rate per fold.
The log ends with the count tables saved in data/rsna_counts_flag10.csv and data/rsna_counts_flag5.csv;
oof_breast.csv and folds.csv are the inputs of experiments/rsna_posthoc.py.

In [ ]:
import os, sys, glob, json, time, shutil, subprocess, threading, queue
import numpy as np, pandas as pd
DRY = os.environ.get("RSNA_DRYRUN") == "1"
EPOCHS = 1 if DRY else 10
ROOT = os.environ.get("RSNA_INPUT", "/kaggle/input"); WORK = os.environ.get("RSNA_WORK", "/kaggle/working/rsna")
os.makedirs(WORK, exist_ok=True)
# Breadth-first scan of the inputs, max depth 4, never descending into the DICOM trees
files, png_count, frontier = [], {}, [ROOT]
for depth in range(4):
    nxt = []
    for d in frontier:
        try: entries = list(os.scandir(d))
        except OSError: continue
        n_png = 0
        for e in entries:
            if e.is_dir(follow_symlinks=False):
                if e.name not in ("train_images", "test_images"): nxt.append(e.path)
            else:
                files.append(e.path); n_png += e.name.endswith(".png")
        png_count[d] = n_png
    frontier = nxt
meta_p = [p for p in files if os.path.basename(p) == "train.csv" and ("rsna" in p.lower() or DRY)]
assert meta_p, "competition train.csv not found: add the RSNA competition as input"
META = sorted(meta_p, key=len)[0]
IMG = max(png_count, key=png_count.get)
assert png_count[IMG] > 1000, "PNG folder not found: add theoviel/rsna-breast-cancer-512-pngs"
IMG = IMG.rstrip("/") + "/"
print("train.csv:", META, "| images:", IMG, max(png_count.values()))

In [ ]:
if not DRY:
    r = subprocess.run("nvidia-smi --query-gpu=name --format=csv,noheader", shell=True, capture_output=True, text=True)
    GPUS = [l for l in r.stdout.splitlines() if l.strip()]; assert GPUS, "no GPU: choose GPU T4 x2"
    import torch, timm
    torch.nn.Conv2d(3, 8, 3).cuda()(torch.randn(2, 3, 64, 64, device="cuda")).sum().item()
    timm.create_model("efficientnet_b0", pretrained=True); print("GPUs", GPUS, "| torch", torch.__version__, "| weights OK")
else:
    GPUS = ["cpu0", "cpu1"]

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold
df = pd.read_csv(META)
need = {"site_id", "patient_id", "image_id", "laterality", "age", "cancer", "density"}
assert need <= set(df.columns), f"missing columns {need - set(df.columns)}"
have = {(int(a), int(b)) for a, b in (os.path.basename(p)[:-4].split("_") for p in glob.glob(IMG + "*.png"))}
df = df[[ (int(a), int(b)) in have for a, b in zip(df.patient_id, df.image_id)]].reset_index(drop=True)
print("images with PNG:", len(df), "| patients:", df.patient_id.nunique())
# consistency checks at breast level
g = df.groupby(["patient_id", "laterality"])
assert (g.cancer.nunique() == 1).all(), "cancer label differs within a breast"
print("breasts with >1 density value:", int((g.density.nunique() > 1).sum()))
pat = df.groupby("patient_id").cancer.max()
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
fold_of = {}
for k, (_, idx) in enumerate(sgkf.split(pat.index.to_numpy()[:, None], pat.to_numpy(), groups=pat.index.to_numpy())):
    for p in pat.index.to_numpy()[idx]: fold_of[p] = k
df["fold"] = df.patient_id.map(fold_of); df.to_csv(f"{WORK}/folds.csv", index=False)
print(df.groupby("fold").agg(images=("image_id", "size"), patients=("patient_id", "nunique"), cancer_imgs=("cancer", "sum")))
B = df.groupby(["site_id", "patient_id", "laterality"]).agg(cancer=("cancer", "max"), density=("density", "first"), age=("age", "first")).reset_index()
print("\nbreasts by site and cancer:\n", B.groupby(["site_id", "cancer"]).size().to_string())
print("\nsite-1 breasts by density and cancer:\n", B[B.site_id == 1].groupby(["density", "cancer"], dropna=False).size().unstack().to_string())

In [ ]:
TRAIN_FOLD_SRC = '"""Train one fold of a breast-cancer image classifier on RSNA 512x512 PNGs and write out-of-fold scores.\nUsage: python train_fold.py --meta folds.csv --img_dir DIR --fold 0 --out OUT [--epochs 10] [--smoke]\n"""\nimport argparse, os, random, time\nimport numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F\nimport timm\nfrom PIL import Image\n\n\nclass DS(torch.utils.data.Dataset):\n    def __init__(self, df, img_dir, train):\n        self.p = [os.path.join(img_dir, f"{a}_{b}.png") for a, b in zip(df.patient_id, df.image_id)]\n        self.y = df.cancer.to_numpy().astype(np.float32); self.train = train\n\n    def __len__(self): return len(self.p)\n\n    def __getitem__(self, i):\n        x = torch.from_numpy(np.asarray(Image.open(self.p[i]).convert("L"), dtype=np.float32) / 255.0)[None]\n        if self.train:\n            if random.random() < .5: x = x.flip(-1)\n            if random.random() < .5: x = x.flip(-2)\n            if random.random() < .5:                                     # small affine\n                ang, sc = random.uniform(-10, 10), random.uniform(0.9, 1.1)\n                th = torch.tensor([[sc * np.cos(np.radians(ang)), -sc * np.sin(np.radians(ang)), random.uniform(-.05, .05)],\n                                   [sc * np.sin(np.radians(ang)), sc * np.cos(np.radians(ang)), random.uniform(-.05, .05)]], dtype=torch.float32)\n                g = F.affine_grid(th[None], x[None].shape, align_corners=False)\n                x = F.grid_sample(x[None], g, align_corners=False)[0]\n            x = (x * random.uniform(0.85, 1.15) + random.uniform(-0.08, 0.08)).clamp(0, 1)\n        x = (x - 0.5) / 0.25\n        return x.repeat(3, 1, 1), self.y[i]\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--meta", required=True); ap.add_argument("--img_dir", required=True)\n    ap.add_argument("--fold", type=int, required=True); ap.add_argument("--out", required=True)\n    ap.add_argument("--epochs", type=int, default=10); ap.add_argument("--bs", type=int, default=32)\n    ap.add_argument("--lr", type=float, default=3e-4); ap.add_argument("--neg_ratio", type=float, default=10)\n    ap.add_argument("--workers", type=int, default=2); ap.add_argument("--seed", type=int, default=42)\n    ap.add_argument("--backbone", default="efficientnet_b0"); ap.add_argument("--no_pretrained", action="store_true")\n    ap.add_argument("--smoke", action="store_true")\n    a = ap.parse_args()\n    random.seed(a.seed + a.fold); np.random.seed(a.seed + a.fold); torch.manual_seed(a.seed + a.fold)\n    dev = "cuda" if torch.cuda.is_available() else "cpu"; amp = dev == "cuda"\n    os.makedirs(a.out, exist_ok=True)\n    df = pd.read_csv(a.meta)\n    tr, te = df[df.fold != a.fold].reset_index(drop=True), df[df.fold == a.fold].reset_index(drop=True)\n    assert not set(tr.patient_id) & set(te.patient_id), "patient leakage across folds"\n    if a.smoke:\n        tr = pd.concat([tr[tr.cancer == 1].head(32), tr[tr.cancer == 0].head(96)]); te = te.head(64)\n    pos, neg = tr[tr.cancer == 1], tr[tr.cancer == 0]\n    model = timm.create_model(a.backbone, pretrained=not a.no_pretrained, num_classes=1).to(dev)\n    opt = torch.optim.AdamW(model.parameters(), lr=a.lr, weight_decay=1e-4)\n    steps_per_epoch = int(np.ceil(len(pos) * (1 + a.neg_ratio) / a.bs))\n    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=a.lr, total_steps=a.epochs * steps_per_epoch, pct_start=0.1)\n    scaler = torch.amp.GradScaler(\'cuda\', enabled=amp)\n    t0 = time.time()\n    for ep in range(a.epochs):\n        n_neg = min(len(neg), int(round(a.neg_ratio * len(pos))))\n        ep_df = pd.concat([pos, neg.sample(n_neg, random_state=a.seed * 100 + ep)]).sample(frac=1, random_state=ep)\n        L = torch.utils.data.DataLoader(DS(ep_df, a.img_dir, True), batch_size=a.bs, shuffle=False,\n                                        num_workers=a.workers, drop_last=True, pin_memory=amp)\n        model.train(); tot, nb = 0.0, 0\n        for x, y in L:\n            x, y = x.to(dev, non_blocking=True), y.to(dev, non_blocking=True)\n            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=amp):\n                loss = F.binary_cross_entropy_with_logits(model(x).squeeze(1), y)\n            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.step(opt); scaler.update()\n            if nb < steps_per_epoch: sched.step()\n            tot += loss.item(); nb += 1\n        print(f"fold {a.fold} epoch {ep} loss {tot / max(nb, 1):.4f} ({(time.time() - t0) / 60:.1f} min)", flush=True)\n    torch.save(model.state_dict(), f"{a.out}/model.pt")\n    # out-of-fold inference, average of original and horizontally flipped image\n    model.eval(); L = torch.utils.data.DataLoader(DS(te, a.img_dir, False), batch_size=2 * a.bs, num_workers=a.workers)\n    outs = []\n    with torch.no_grad():\n        for x, _ in L:\n            x = x.to(dev)\n            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=amp):\n                s = (torch.sigmoid(model(x).float()) + torch.sigmoid(model(x.flip(-1)).float())) / 2\n            outs.append(s.squeeze(1).cpu().numpy())\n    te["score"] = np.concatenate(outs)\n    te[["patient_id", "image_id", "laterality", "fold", "score"]].to_csv(f"{a.out}/oof.csv", index=False)\n    print(f"fold {a.fold} done: {len(te)} images scored in {(time.time() - t0) / 60:.1f} min", flush=True)\n\n\nif __name__ == "__main__":\n    main()\n'
open(f'{WORK}/train_fold.py', 'w').write(TRAIN_FOLD_SRC); print('train_fold.py written', len(TRAIN_FOLD_SRC), 'chars')

In [ ]:

def run(fold, gpu, epochs, out, extra=()):
    if os.path.exists(f"{out}/oof.csv"): print("skip", out); return 0
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu), PYTHONUNBUFFERED="1")
    c = [sys.executable, f"{WORK}/train_fold.py", "--meta", f"{WORK}/folds.csv", "--img_dir", IMG, "--fold", str(fold),
         "--out", out, "--epochs", str(epochs), *extra]
    with open(f"{out}.log", "w") as log:
        return subprocess.run(c, env=env, stdout=log, stderr=subprocess.STDOUT).returncode
X = ("--no_pretrained", "--workers", "0", "--bs", "4") if DRY else ()
rc = run(0, 0, 1, f"{WORK}/smoke", ("--smoke",) + X); print(open(f"{WORK}/smoke.log").read()[-1500:]); assert rc == 0, "smoke failed"


In [ ]:
q = queue.Queue(); [q.put(k) for k in range(5)]; status = {}; T0 = time.time()
def worker(gpu):
    while True:
        try: k = q.get_nowait()
        except queue.Empty: return
        print(f"[{(time.time()-T0)/60:5.1f} min] fold {k} start on GPU {gpu}", flush=True)
        status[k] = run(k, gpu, EPOCHS, f"{WORK}/fold{k}", X)
        print(f"[{(time.time()-T0)/60:5.1f} min] fold {k} rc={status[k]}", flush=True)
        print(open(f"{WORK}/fold{k}.log").read()[-600:], flush=True)
ths = [threading.Thread(target=worker, args=(g,)) for g in range(2)]; [t.start() for t in ths]; [t.join() for t in ths]
assert all(v == 0 for v in status.values()) and len(status) == 5, status

In [ ]:
from sklearn.metrics import roc_auc_score
oof = pd.concat([pd.read_csv(f"{WORK}/fold{k}/oof.csv") for k in range(5)])
d = df.merge(oof[["patient_id", "image_id", "score"]], on=["patient_id", "image_id"], how="inner")
assert len(d) == len(df), (len(d), len(df))
B = d.groupby(["site_id", "patient_id", "laterality", "fold"]).agg(cancer=("cancer", "max"), density=("density", "first"),
                                                                 age=("age", "first"), score=("score", "mean")).reset_index()
B.to_csv(f"{WORK}/oof_breast.csv", index=False)
print("breast-level AUC all:", round(roc_auc_score(B.cancer, B.score), 4),
      "| site1:", round(roc_auc_score(B[B.site_id == 1].cancer, B[B.site_id == 1].score), 4),
      "| site2:", round(roc_auc_score(B[B.site_id == 2].cancer, B[B.site_id == 2].score), 4))
for fpr in (0.10, 0.05):
    thr = {k: np.quantile(B[(B.fold == k) & (B.cancer == 0)].score, 1 - fpr) for k in range(5)}
    B[f"flag{int(fpr*100)}"] = (B.score > B.fold.map(thr)).astype(int)
B["age_band"] = pd.cut(B.age, [0, 39.999, 49.999, 59.999, 69.999, 200], labels=["<40", "40-49", "50-59", "60-69", "70+"])
S1 = B[(B.site_id == 1) & B.density.notna()]
print(f"\nsite-1 breasts with density: {len(S1)} ({int(S1.cancer.sum())} cancer); missing age: {int(S1.age.isna().sum())}")
for f in ("flag10", "flag5"):
    print(f"\n===== COUNTS {f}: site 1, by age band x density x cancer (n, flagged) =====")
    t = S1.groupby(["age_band", "density", "cancer"], observed=True)[f].agg(["size", "sum"]).reset_index()
    print(t.to_string(index=False))
    print(f"--- {f}: overall site-1 by density ---")
    print(S1.groupby(["density", "cancer"])[f].agg(["size", "sum", "mean"]).round(4).to_string())
print("\nexact age distribution of site-1 cancers with density:", S1[S1.cancer == 1].age.describe().round(1).to_dict())
import zipfile
with zipfile.ZipFile("/kaggle/working/rsna_outputs.zip" if not DRY else f"{WORK}/rsna_outputs.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for p in [f"{WORK}/oof_breast.csv", f"{WORK}/folds.csv"] + glob.glob(f"{WORK}/fold*.log"):
        z.write(p, os.path.basename(p))
print("done")